# USD Curve Parity Walkthrough

This notebook shows the USD parity trace as one connected workflow:

1. ORE input chain selection
2. USD discount curve identity and dependency
3. Segment quotes and conventions
4. Native interpolation nodes from `todaysmarketcalibration.csv`
5. Report grid points from `curves.csv`
6. How those pieces fit together


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name and ROOT.name != "Tools":
    ROOT = ROOT.parent
if ROOT.name != "Tools":
    raise RuntimeError("Run this notebook from somewhere under the Engine workspace")

PYRUNNER = ROOT / "PythonOreRunner"
if str(PYRUNNER) not in sys.path:
    sys.path.insert(0, str(PYRUNNER))

from ore_curve_fit_parity.curve_trace import trace_usd_curve_from_ore
from ore_curve_fit_parity.interpolation import (
    build_log_linear_discount_interpolator,
)

ORE_XML = PYRUNNER / "parity_artifacts" / "multiccy_benchmark_final" / "cases" / "flat_USD_5Y_B" / "Input" / "ore.xml"
TRACE_JSON = PYRUNNER / "parity_artifacts" / "usd_curve_trace" / "usd3m_flat_usd_5y_b.json"


In [ ]:
trace = trace_usd_curve_from_ore(ORE_XML)
TRACE_JSON.parent.mkdir(parents=True, exist_ok=True)
TRACE_JSON.write_text(json.dumps(trace, indent=2, sort_keys=True) + "\n", encoding="utf-8")

segment_rows = []
for segment in trace["segment_alignment"]:
    convention = trace["conventions"].get(segment["conventions"], {})
    for quote in segment["quotes"]:
        ore_pillar = quote.get("ore_pillar") or {}
        segment_rows.append(
            {
                "segment_type": segment["type"],
                "convention_id": segment["conventions"],
                "convention_type": convention.get("convention_type", ""),
                "quote_key": quote["quote_key"],
                "market_quote": quote["quote_value"],
                "pillar_date": ore_pillar.get("date", ""),
                "pillar_time": float(ore_pillar.get("time", 0.0) or 0.0),
                "pillar_df": float(ore_pillar.get("discountFactor", 0.0) or 0.0),
                "pillar_zero": float(ore_pillar.get("zeroRate", 0.0) or 0.0),
                "matched": bool(ore_pillar),
            }
        )

segment_df = pd.DataFrame(segment_rows)
native_nodes_df = pd.DataFrame(trace["ore_calibration_trace"]["pillars"]).rename(
    columns={
        "date": "calendar_date",
        "time": "time",
        "discountFactor": "df",
        "zeroRate": "zero_rate",
        "forwardRate": "forward_rate",
        "mdQuote": "market_quote",
    }
)
for col in ["time", "df", "zero_rate", "forward_rate", "market_quote", "calibrationError"]:
    if col in native_nodes_df.columns:
        native_nodes_df[col] = pd.to_numeric(native_nodes_df[col], errors="coerce")

curve_df = pd.DataFrame(
    {
        "calendar_date": trace["ore_curve_points"]["calendar_dates"],
        "time": trace["ore_curve_points"]["times"],
        "df": trace["ore_curve_points"]["dfs"],
    }
)

def build_dense_display_grid(native_nodes_df, n_points=1200):
    interp_variable = trace["curve_config"]["interpolation_variable"]
    interp_method = trace["curve_config"]["interpolation_method"]
    if interp_variable != "Discount" or interp_method != "LogLinear":
        raise ValueError(f"Unsupported display-grid interpolation: {interp_variable} / {interp_method}")
    x = native_nodes_df["time"].to_numpy(dtype=float)
    y = native_nodes_df["df"].to_numpy(dtype=float)
    curve_fn = build_log_linear_discount_interpolator(x.tolist(), y.tolist())
    dense_x = np.linspace(float(x.min()), float(x.max()), int(n_points))
    dense_df = np.array([curve_fn(float(t)) for t in dense_x], dtype=float)
    return pd.DataFrame({"time": dense_x, "df": dense_df})

dense_curve_df = build_dense_display_grid(native_nodes_df)

summary = pd.Series(
    {
        "configuration_id": trace["configuration_id"],
        "curve_handle": trace["curve_handle"],
        "curve_name": trace["curve_name"],
        "curve_id": trace["curve_config"]["curve_id"],
        "discount_dependency": trace["curve_config"]["discount_curve"],
        "interpolation_variable": trace["curve_config"]["interpolation_variable"],
        "interpolation_method": trace["curve_config"]["interpolation_method"],
        "native_interpolation_nodes": len(native_nodes_df),
        "report_grid_points": len(curve_df),
        "dense_display_points": len(dense_curve_df),
        "matched_segment_quotes": int(segment_df["matched"].sum()),
    }
)
summary


In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.8))
ax.axis("off")

boxes = [
    (0.04, "ore.xml", "simulation=libor"),
    (0.28, "todaysmarket.xml", trace["curve_handle"]),
    (0.52, "curveconfig.xml", trace["curve_config"]["curve_id"]),
    (0.76, "conventions.xml", "segment conventions"),
]

for x, title, detail in boxes:
    ax.text(
        x,
        0.55,
        f"{title}\n{detail}",
        ha="center",
        va="center",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.5", fc="#eef3f7", ec="#355070", lw=1.5),
        transform=ax.transAxes,
    )

for start, end in [(0.13, 0.19), (0.37, 0.43), (0.61, 0.67)]:
    ax.annotate(
        "",
        xy=(end, 0.55),
        xytext=(start, 0.55),
        arrowprops=dict(arrowstyle="->", lw=2, color="#355070"),
        xycoords=ax.transAxes,
    )

ax.set_title("How the USD3M parity trace is assembled")
plt.show()


## Native Interpolation Nodes vs Report Grid

For this curve, ORE is configured with `InterpolationVariable = Discount` and `InterpolationMethod = LogLinear`.

- `native_nodes_df` below comes from `todaysmarketcalibration.csv` and represents the native pillar discount factors ORE calibrates and interpolates between.
- `curve_df` comes from `curves.csv` and is the original ORE report grid.
- `dense_curve_df` is a finer display grid derived from the native nodes using the same `Discount + LogLinear` interpolation rule as the curve config.


In [ ]:
native_nodes_df[["quote_key", "calendar_date", "time", "df", "zero_rate", "forward_rate", "market_quote"]].head(22)


In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.8))
ax.plot(dense_curve_df["time"], dense_curve_df["df"], lw=2.2, color="#1d3557", label="Dense display grid")
ax.scatter(curve_df["time"], curve_df["df"], s=14, color="#4f772d", alpha=0.7, label="curves.csv report grid")
ax.scatter(native_nodes_df["time"], native_nodes_df["df"], s=42, color="#d62828", label="Native interpolation nodes")
ax.set_title("Native interpolation nodes vs fine display grid")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Discount factor")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


In [ ]:
segment_summary = (
    segment_df.groupby(["segment_type", "convention_id", "convention_type"], as_index=False)
    .agg(quote_count=("quote_key", "count"), matched_pillars=("matched", "sum"))
)
segment_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].barh(segment_summary["segment_type"], segment_summary["quote_count"], color="#457b9d")
axes[0].set_title("Quotes per USD3M segment")
axes[0].set_xlabel("Quote count")

axes[1].barh(segment_summary["segment_type"], segment_summary["matched_pillars"], color="#2a9d8f")
axes[1].set_title("Matched ORE pillars per segment")
axes[1].set_xlabel("Matched pillars")

plt.tight_layout()
plt.show()


In [ ]:
colors = {"Deposit": "#e76f51", "FRA": "#f4a261", "Swap": "#264653"}
fig, ax = plt.subplots(figsize=(10, 4.5))
for segment_type, group in segment_df.groupby("segment_type"):
    ax.scatter(group["pillar_time"], group["pillar_zero"], s=50, label=segment_type, color=colors.get(segment_type, "#777777"))

ax.set_title("ORE calibration pillars by segment type")
ax.set_xlabel("Pillar time (years)")
ax.set_ylabel("Zero rate")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(dense_curve_df["time"], dense_curve_df["df"], lw=2.2, color="#1d3557", label="Dense display grid")
ax.scatter(curve_df["time"], curve_df["df"], s=12, color="#4f772d", alpha=0.55, label="curves.csv report grid")
ax.scatter(native_nodes_df["time"], native_nodes_df["df"], s=42, color="#d62828", label="Native interpolation nodes")
for segment_type, group in segment_df.groupby("segment_type"):
    ax.scatter(group["pillar_time"], group["pillar_df"], s=18, alpha=0.65, label=f"{segment_type} nodes", color=colors.get(segment_type, "#777777"))

ax.set_title("USD3M final curve with fine display grid, native nodes, and segment labeling")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Discount factor")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2)
plt.show()


In [ ]:
segment_df[["segment_type", "convention_id", "quote_key", "market_quote", "pillar_date", "pillar_time", "pillar_df", "pillar_zero"]].head(20)
